In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfTransformer, TfidfVectorizer, CountVectorizer
csv_path = r"C:\Users\Mumba Mumba\Desktop\NLP\projects\book searcher\data\books_dataset.csv"
df = pd.read_csv(csv_path)
df.head()


,isbn13,title,author,description,genres,avg_rating,ratings_count,publisher,format,pages,pub_year,series_name,series_position
0,9780439023481,"The Hunger Games (The Hunger Games, #1)",Suzanne Collins,Winning will make you famous.\nLosing means ce...,"[""young-adult"", ""fiction"", ""fantasy, paranorma...",4.34,4899965.0,Scholastic Press,Hardcover,374.0,2008.0,The Hunger Games,1.0
1,9780439554930,Harry Potter and the Sorcerer's Stone (Harry P...,J.K. Rowling,Harry Potter's life is miserable. His parents ...,"[""fantasy, paranormal"", ""young-adult"", ""fictio...",4.45,4765497.0,Scholastic Inc,Hardcover,320.0,1997.0,Harry Potter,1.0
2,9780316015844,"Twilight (Twilight, #1)",Stephenie Meyer,About three things I was absolutely positive.\...,"[""fantasy, paranormal"", ""young-adult"", ""fictio...",3.57,3941381.0,"Little, Brown and Company",Paperback,498.0,2006.0,Twilight,1.0
3,9781649374042,"Fourth Wing (The Empyrean, #1)",Rebecca Yarros,A #1 New York Times bestseller • TV series in ...,"[""Fantasy"", ""Romance"", ""Romantasy"", ""Dragons"",...",4.56,3818964.0,Entangled: Red Tower Books,NaN,517.0,2023.0,NaN,NaN
4,9780061120084,To Kill a Mockingbird,Harper Lee,The unforgettable novel of a childhood in a sl...,"[""fiction"", ""history, historical fiction, biog...",4.26,3255518.0,Harper Perennial Modern Classics,Paperback,324.0,2006.0,To Kill a Mockingbird,NaN


In [2]:
df.columns.tolist()

['isbn13',
 'title',
 'author',
 'description',
 'genres',
 'avg_rating',
 'ratings_count',
 'publisher',
 'format',
 'pages',
 'pub_year',
 'series_name',
 'series_position']

In [3]:
df.isnull().sum().to_frame()

,0
isbn13,0
title,0
author,1420
description,5514
genres,14775
avg_rating,16745
ratings_count,17631
publisher,10018
format,29392
pages,22165


In [4]:
missing_val_incol = ['author', 'description', 'genres', 'avg_rating', 'ratings_count', 'publisher', 'format', 'pages', 'pub_year', 'series_name', 'series_position']

In [5]:

#fill missing values so that string concatenation does not fail
missing_val_incol = ['author', 'description', 'genres', 'avg_rating', 'ratings_count', 'publisher', 'format', 'pages', 'pub_year', 'series_name', 'series_position']

for col in missing_val_incol:
    if col in df.columns:
        df[col] = df[col].fillna("")

In [6]:
#combine description and genres for richer thematic matching
df['vibe_text'] = df['description'] + " " + df['genres']

In [7]:
df = df[df['vibe_text'].str.strip() != ""].reset_index(drop=False)

In [8]:
#vectorize with tfidf
vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=25000,
    ngram_range= (1,2)
)
tfidf_matrix = vectorizer.fit_transform(df['vibe_text'])

In [9]:
tfidf_matrix.shape

(84031, 25000)

In [14]:
def recommend_by_vibe(prompt: str, top_n: int = 5):
    query_vec = vectorizer.transform([prompt])
    scores = cosine_similarity(query_vec, tfidf_matrix).flatten()

    top_indices = np.argsort(scores)[::-1][top_n]
    results = df.iloc[top_indices][['title', 'author', 'genres']].copy()
    results['similsrity_score'] = scores[top_indices].round(3)
    return results

In [15]:
def recommend_by_title(book_title: str, top_n: int = 5):
    # Case-insensitive partial match to find the book index
    matches = df[df["title"].str.lower().str.contains(book_title.lower(), regex=False)]
    if matches.empty:
        return f"No book found matching '{book_title}'."
    
    idx = matches.index[0]
    print(f"Matched Book: {df.loc[idx, 'title']} by {df.loc[idx, 'author']}\n")
    
    scores = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
    # Skip index 0 after sorting because a book is always 1.0 similar to itself
    top_indices = np.argsort(scores)[::-1][1 : top_n + 1]
    
    results = df.iloc[top_indices][["title", "author", "genres"]].copy()
    results["similarity_score"] = scores[top_indices].round(4)
    return results

In [16]:
if __name__ == "__main__":
    print("\n--- Vibe Search ---")
    print(recommend_by_vibe("lonely astronaut stranded on a desolate planet solving puzzles"))
    
    print("\n--- Title Search ---")
    print(recommend_by_title("The Hobbit"))


--- Vibe Search ---
title                            Return From the Stars
author                                   Stanislaw Lem
genres              ["fiction", "fantasy, paranormal"]
similsrity_score                                 0.198
Name: 37785, dtype: object

--- Title Search ---
Matched Book: The Hobbit by J.R.R. Tolkien

                           title  ... similarity_score
48526                 The Hobbit  ...           0.5463
42362                 The Hobbit  ...           0.3909
19055                 The Hobbit  ...           0.3871
611    The Hobbit: Graphic Novel  ...           0.3594
31246  The Hobbit: Graphic Novel  ...           0.3517

[5 rows x 4 columns]
